In [562]:
import subprocess
import sys
import os
import warnings

In [ ]:
COMPANY_TICKER = "GOOGL" #change to the ticker of the company you want to analyze each time
COMPANY_NAME = "Alphabet Inc." #change to the name of the company you want to analyze each time
PEER_TICKERS = ["META", "MSFT"] #change to the tickers of the peers of the company you want to analyze each time
NEWS_DAYS_BACK = 5 
# OUTPUT_BASE_DIR = "../output"  #original
# Output root directory: Directly write to assign2_reports/< Experiment name >/. When changing companies, only the above three lines need to be modified. There is no need to copy manually anymore.
# When switching experiments, such as exp1_model_combo, only change EXPERIMENT_FOLDER. To switch back to the default output folder, use "../output".
EXPERIMENT_FOLDER = "exp6_gpt4o"
OUTPUT_BASE_DIR = f"../../../assign2_reports/{EXPERIMENT_FOLDER}"

CONFIG_FILE = "../config/config.ini"
# Set True to append optional "Analyst commentary" in earnings_update.md (uses API keys in CONFIG_FILE).
EARNINGS_UPDATE_USE_LLM_SUMMARY = False
# RUN_TRACK_B：是否生成 Track B 三件套（thesis_tracker / catalyst_calendar / earnings_update）并写入 HTML/PDF。
# True：Step2 前生成 earnings 供 HTML；Step3 含 Track B 预步骤 + PDF；Step4 再刷新三件套。
# False：基线模型 — 仍生成 HTML + PDF（Step3 只跑 PDF），不跑上述脚本；Step4 跳过。若 analysis/ 里仍有旧 .md，PDF 可能读到，干净基线请删对应三文件后重跑。
RUN_TRACK_B = False

# The company and peer company used each time
COMPANIES = [
    {
        "ticker": "NVDA",
        "name": "NVIDIA Corporation",
        "peers": ["AMD", "INTC"]
    },
    {
        "ticker": "AMD",
        "name": "Advanced Micro Devices, Inc.",
        "peers": ["NVDA", "INTC"]
    },
    {
        "ticker": "INTC",
        "name": "Intel Corporation",
        "peers": ["NVDA", "AMD"]
    },
    {
        "ticker": "AAPL",
        "name": "Apple Inc.",
        "peers": ["MSFT", "GOOGL"]
    },
    {
        "ticker": "GOOGL",
        "name": "Alphabet Inc.",
        "peers": ["META", "MSFT"]
    }
]

In [564]:
print(f"🚀 Starting FMP API-based equity research report generation for {COMPANY_NAME} ({COMPANY_TICKER})")
print("=" * 80)

🚀 Starting FMP API-based equity research report generation for Alphabet Inc. (GOOGL)


In [565]:
command = [
    sys.executable,
    "generate_financial_analysis.py",
    "--company-ticker", COMPANY_TICKER,
    "--company-name", COMPANY_NAME,
    "--config-file", CONFIG_FILE,
    "--years-limit", "5",
    "--peer-tickers"] + PEER_TICKERS + [
    "--generate-text-sections",  # Enable AI text generation
    "--enable-sensitivity-analysis",  # Writes sensitivity_analysis.json for the HTML report
    "--enable-catalyst-analysis",  # Writes catalyst_analysis.json when company news is available
    "--output-dir", f"{OUTPUT_BASE_DIR}/{COMPANY_TICKER}/analysis",
    "--news-days-back", str(NEWS_DAYS_BACK),  # NEW: Add news days parameter
    "--revenue-growth-2025", "0.05",   # 5% growth assumption for 2025E
    "--revenue-growth-2026", "0.06",   # 6% growth assumption for 2026E  
    "--revenue-growth-2027", "0.04",   # 4% growth assumption for 2027E
    "--margin-improvement", "0.01",    # 1% annual margin improvement
    "--sga-margin-improvement", "-0.005"  # 0.5% SG&A efficiency gain
]

try:
    result = subprocess.run(command, capture_output=True, text=True, check=True)
    print("✅ Financial analysis completed successfully!")
    print("\nOutput:", result.stdout)
    if result.stderr:
        print("Warnings:", result.stderr)
except subprocess.CalledProcessError as e:
    print(f"❌ Financial analysis failed with return code {e.returncode}")
    print("Output:", e.stdout)
    print("Error:", e.stderr)
    sys.exit(1)

✅ Financial analysis completed successfully!

Output: Output will be saved to: ../../../assign2_reports/exp6_gpt4o/GOOGL/analysis
Text outputs will be saved to: ../../../assign2_reports/exp6_gpt4o/GOOGL/analysis
Using OpenAI base URL: https://api.openai.com/v1
Using model: gpt-4o
Starting FMP API-based financial analysis for Alphabet Inc. (GOOGL)
Fetching financial data from FMP API...
Fetching comprehensive financial data for GOOGL...
Successfully fetched financial data from FMP API
Retrieved 5 years of income statement data
Available years: [2025, 2024, 2023, 2022, 2021]
Processing historical financial metrics...

Historical Metrics Extracted from API:
                metrics           2025A           2024A           2023A           2022A           2021A
0               Revenue  402963000000.0  350018000000.0  307394000000.0  282836000000.0  257637000000.0
1    Cost of Operations  162535000000.0  146306000000.0  133332000000.0  126203000000.0  110939000000.0
2                  SG&A  

In [566]:
import subprocess
import sys
import os

In [567]:
analysis_dir = f"{OUTPUT_BASE_DIR}/{COMPANY_TICKER}/analysis"
report_dir = f"{OUTPUT_BASE_DIR}/{COMPANY_TICKER}/report"
print("\n📄 Step 2: Creating Equity Research Report...")
print("-" * 80)



📄 Step 2: Creating Equity Research Report...
--------------------------------------------------------------------------------


In [568]:
# Step 2 的 HTML 会读 earnings_update.md；RUN_TRACK_B=True 时在 Step 2 前生成。False=基线模型不生成；Step3 仍会出 PDF（无 Track B 预步骤），Step4 跳过。
if RUN_TRACK_B:
    print("\n📌 Pre-Step 2: Earnings Update (for HTML)...")
    earnings_update_pre_html = [
        sys.executable,
        "generate_earnings_update.py",
        "--company-ticker", COMPANY_TICKER,
        "--company-name", COMPANY_NAME,
        "--analysis-dir", analysis_dir,
    ]
    if EARNINGS_UPDATE_USE_LLM_SUMMARY:
        earnings_update_pre_html += ["--use-llm-summary", "--config-file", CONFIG_FILE]
    try:
        _eu_pre = subprocess.run(earnings_update_pre_html, capture_output=True, text=True, check=True)
        print(_eu_pre.stdout.strip() or "✅ Earnings Update ready for HTML.")
        if _eu_pre.stderr:
            print("Earnings Update warnings:", _eu_pre.stderr)
    except subprocess.CalledProcessError as e:
        print("⚠️ Earnings Update failed before HTML; section may be empty. Re-run after fixing.")
        print("Output:", e.stdout)
        print("Error:", e.stderr)
else:
    print("\n⏭️ Pre-Step 2: skipped (RUN_TRACK_B=False). ")



command = [
    sys.executable,
    "create_equity_report.py",
    "--company-ticker", COMPANY_TICKER,
    "--company-name", COMPANY_NAME,
    "--analysis-csv", f"{analysis_dir}/financial_metrics_and_forecasts.csv",
    "--tagline-file", f"{analysis_dir}/tagline.txt",
    "--company-overview-file", f"{analysis_dir}/company_overview.txt",
    "--investment-overview-file", f"{analysis_dir}/investment_overview.txt",
    "--valuation-overview-file", f"{analysis_dir}/valuation_overview.txt",
    "--risks-file", f"{analysis_dir}/risks.txt",
    "--competitor-analysis-file", f"{analysis_dir}/competitor_analysis.txt", # change from peer_ebitda_comparison.csv to competitor_analysis.txt
    "--major-takeaways-file", f"{analysis_dir}/major_takeaways.txt",
    "--news-summary-file", f"{analysis_dir}/news_summary.txt",  # NEW: Add news summary file
    "--peer-ev-ebitda-csv", f"{analysis_dir}/peer_ev_ebitda_comparison.csv",
    "--peer-ebitda-csv", f"{analysis_dir}/peer_ebitda_comparison.csv",
    "--ratios-csv", f"{analysis_dir}/ratios_raw_data.csv",
    "--sensitivity-analysis-file", f"{analysis_dir}/sensitivity_analysis.json",
    "--catalyst-analysis-file", f"{analysis_dir}/catalyst_analysis.json",
    "--enable-text-regeneration",
    "--output-dir", report_dir,
    "--config-file", CONFIG_FILE,
    # Note: All market data metrics will be auto-fetched from FMP API!
    # No need to manually specify share-price, target-price, market-cap, etc.
]
if not RUN_TRACK_B:
    command.append("--omit-earnings-update-html")

try:
    result = subprocess.run(command, capture_output=True, text=True, check=True)
    print("✅ Equity report created successfully!")
    print("\nOutput:", result.stdout)
    if result.stderr:
        print("Warnings:", result.stderr)
except subprocess.CalledProcessError as e:
    print(f"❌ Equity report generation failed with return code {e.returncode}")
    print("Output:", e.stdout)
    print("Error:", e.stderr)
    sys.exit(1)


⏭️ Pre-Step 2: skipped (RUN_TRACK_B=False). 
✅ Equity report created successfully!

Output: HTML reports will be saved to: ../../../assign2_reports/exp6_gpt4o/GOOGL/report
✅ OpenAI API key loaded for text regeneration
Using OpenAI base URL for regeneration: https://api.openai.com/v1
Using model for regeneration: gpt-4o
Auto-fetching market data for GOOGL...
Fetching comprehensive company metrics for GOOGL...
Successfully fetched metrics for GOOGL
✅ Successfully auto-fetched market data

📊 Market Data Summary for GOOGL:
  Share Price: $349.94 (auto-fetched)
  Target Price: $375.00 (auto-fetched)
  Rating: Overweight (auto-fetched)
  Market Cap: $4,233.38B (auto-fetched)
  Sector: Communication Services (auto-fetched)
📖 Loading and processing text content...
📝 Processing tagline...
✅ tagline validation passed (429 chars)
📝 Processing company_overview...
✅ company_overview validation passed (2475 chars)
📝 Processing investment_overview...
✅ investment_overview validation passed (1693 cha

In [569]:
def _run_step3_pdf():
    """Step 3: always generate PDF. Track B pre-steps (thesis / calendar / earnings md) only if RUN_TRACK_B."""
    if RUN_TRACK_B:
        print("\n📌 Pre-step: Ensuring Thesis Tracker exists for PDF...")
        tracker_command = [
            sys.executable,
            "generate_thesis_tracker.py",
            "--company-ticker", COMPANY_TICKER,
            "--company-name", COMPANY_NAME,
            "--analysis-dir", analysis_dir,
        ]
        try:
            tracker_result = subprocess.run(tracker_command, capture_output=True, text=True, check=True)
            print(tracker_result.stdout.strip() or "✅ Thesis Tracker ready.")
            if tracker_result.stderr:
                print("Tracker warnings:", tracker_result.stderr)
        except subprocess.CalledProcessError as e:
            print("⚠️ Thesis Tracker pre-step failed; continuing with PDF generation.")
            print("Output:", e.stdout)
            print("Error:", e.stderr)

        print("\n📌 Pre-step: Building Catalyst Calendar (timeline + types)...")
        calendar_command = [
            sys.executable,
            "generate_catalyst_calendar.py",
            "--company-ticker", COMPANY_TICKER,
            "--company-name", COMPANY_NAME,
            "--analysis-dir", analysis_dir,
        ]
        try:
            cal_result = subprocess.run(calendar_command, capture_output=True, text=True, check=True)
            print(cal_result.stdout.strip() or "✅ Catalyst Calendar ready.")
            if cal_result.stderr:
                print("Calendar warnings:", cal_result.stderr)
        except subprocess.CalledProcessError as e:
            print("⚠️ Catalyst Calendar pre-step failed; continuing with PDF generation.")
            print("Output:", e.stdout)
            print("Error:", e.stderr)

        print("\n📌 Pre-step: Earnings Update (earnings-analysis skill subset)...")
        earnings_update_command = [
            sys.executable,
            "generate_earnings_update.py",
            "--company-ticker", COMPANY_TICKER,
            "--company-name", COMPANY_NAME,
            "--analysis-dir", analysis_dir,
        ]
        if EARNINGS_UPDATE_USE_LLM_SUMMARY:
            earnings_update_command += ["--use-llm-summary", "--config-file", CONFIG_FILE]
        try:
            eu_result = subprocess.run(earnings_update_command, capture_output=True, text=True, check=True)
            print(eu_result.stdout.strip() or "✅ Earnings Update ready.")
            if eu_result.stderr:
                print("Earnings Update warnings:", eu_result.stderr)
        except subprocess.CalledProcessError as e:
            print("⚠️ Earnings Update pre-step failed; continuing with PDF generation.")
            print("Output:", e.stdout)
            print("Error:", e.stderr)
    else:
        print("\n⏭️ Track B pre-steps skipped (RUN_TRACK_B=False). PDF still runs (base model pipeline).")
        print("   (If old thesis_tracker.md / catalyst_calendar.md / earnings_update.md remain in analysis/, PDF may still load them — delete those files for a clean baseline.)")

    pdf_command = [
        sys.executable,
        "generate_pdf_report.py",
        "--company-ticker", COMPANY_TICKER,
        "--company-name", COMPANY_NAME,
        "--analysis-dir", analysis_dir,
        "--output-dir", report_dir,
        "--config-file", CONFIG_FILE,
    ]

    print("\n📕 Step 3: Generating PDF Report...")
    print("-" * 80)

    try:
        result = subprocess.run(pdf_command, capture_output=True, text=True, check=True)
        print("✅ PDF report created successfully!")
        print("\nOutput:", result.stdout)
        if result.stderr:
            print("Warnings:", result.stderr)
    except subprocess.CalledProcessError as e:
        print(f"❌ PDF report generation failed with return code {e.returncode}")
        print("Output:", e.stdout)
        print("Error:", e.stderr)
        sys.exit(1)


_run_step3_pdf()



⏭️ Track B pre-steps skipped (RUN_TRACK_B=False). PDF still runs (base model pipeline).
   (If old thesis_tracker.md / catalyst_calendar.md / earnings_update.md remain in analysis/, PDF may still load them — delete those files for a clean baseline.)

📕 Step 3: Generating PDF Report...
--------------------------------------------------------------------------------
✅ PDF report created successfully!

Output: 
📄 PROFESSIONAL EQUITY RESEARCH PDF GENERATOR
Company: Alphabet Inc. (GOOGL)
Analysis Dir: ../../../assign2_reports/exp6_gpt4o/GOOGL/analysis
Output Dir: ../../../assign2_reports/exp6_gpt4o/GOOGL/report

📥 Loading analysis data...
✅ Loaded financial metrics from ../../../assign2_reports/exp6_gpt4o/GOOGL/analysis/financial_metrics_and_forecasts.csv
✅ Loaded ratios data from ../../../assign2_reports/exp6_gpt4o/GOOGL/analysis/ratios_raw_data.csv
✅ Loaded peer EBITDA data
✅ Loaded peer EV/EBITDA data
✅ Loaded tagline
✅ Loaded company_overview
✅ Loaded investment_overview
✅ Loaded valua

In [570]:
def _run_step4_track_b():
    # Step 4: Track B artifacts — Thesis Tracker + Catalyst Calendar + Earnings Update (optional LLM: EARNINGS_UPDATE_USE_LLM_SUMMARY)
    thesis_tracker_command = [
        sys.executable,
        "generate_thesis_tracker.py",
        "--company-ticker", COMPANY_TICKER,
        "--company-name", COMPANY_NAME,
        "--analysis-dir", analysis_dir,
    ]

    print("\n📌 Step 4a: Thesis Tracker (thesis_tracker.md)...")
    print("-" * 80)

    try:
        result = subprocess.run(thesis_tracker_command, capture_output=True, text=True, check=True)
        print(result.stdout.strip() or "✅ Thesis Tracker step finished.")
        if result.stderr:
            print("Warnings:", result.stderr)
    except subprocess.CalledProcessError as e:
        print(f"❌ Thesis Tracker failed with return code {e.returncode}")
        print("Output:", e.stdout)
        print("Error:", e.stderr)
        sys.exit(1)

    calendar_command = [
        sys.executable,
        "generate_catalyst_calendar.py",
        "--company-ticker", COMPANY_TICKER,
        "--company-name", COMPANY_NAME,
        "--analysis-dir", analysis_dir,
    ]

    print("\n📌 Step 4b: Catalyst Calendar (catalyst_calendar.md)...")
    print("-" * 80)

    try:
        result = subprocess.run(calendar_command, capture_output=True, text=True, check=True)
        print(result.stdout.strip() or "✅ Catalyst Calendar step finished.")
        if result.stderr:
            print("Warnings:", result.stderr)
    except subprocess.CalledProcessError as e:
        print(f"⚠️ Catalyst Calendar failed with return code {e.returncode}")
        print("Output:", e.stdout)
        print("Error:", e.stderr)

    earnings_update_command = [
        sys.executable,
        "generate_earnings_update.py",
        "--company-ticker", COMPANY_TICKER,
        "--company-name", COMPANY_NAME,
        "--analysis-dir", analysis_dir,
    ]
    if EARNINGS_UPDATE_USE_LLM_SUMMARY:
        earnings_update_command += ["--use-llm-summary", "--config-file", CONFIG_FILE]

    print("\n📌 Step 4c: Earnings Update (earnings_update.md)...")
    print("-" * 80)

    try:
        result = subprocess.run(earnings_update_command, capture_output=True, text=True, check=True)
        print(result.stdout.strip() or "✅ Earnings Update step finished.")
        if result.stderr:
            print("Warnings:", result.stderr)
    except subprocess.CalledProcessError as e:
        print(f"⚠️ Earnings Update failed with return code {e.returncode}")
        print("Output:", e.stdout)
        print("Error:", e.stderr)


if RUN_TRACK_B:
    _run_step4_track_b()
else:
    print('\n⏭️ Step 4 (Thesis / Calendar / Earnings) skipped (RUN_TRACK_B=False; model-only).')



⏭️ Step 4 (Thesis / Calendar / Earnings) skipped (RUN_TRACK_B=False; model-only).


In [571]:
'''
COMPANIES = [
    {
        "ticker": "NVDA",
        "name": "NVIDIA Corporation",
        "peers": ["AMD", "INTC"]
    },
    {
        "ticker": "AMD",
        "name": "Advanced Micro Devices, Inc.",
        "peers": ["NVDA", "INTC"]
    },
    {
        "ticker": "INTC",
        "name": "Intel Corporation",
        "peers": ["NVDA", "AMD"]
    },
    {
        "ticker": "AAPL",
        "name": "Apple Inc.",
        "peers": ["MSFT", "GOOGL"]
    },
    {
        "ticker": "GOOGL",
        "name": "Alphabet Inc.",
        "peers": ["META", "MSFT"]
    }
]
'''

'\nCOMPANIES = [\n    {\n        "ticker": "NVDA",\n        "name": "NVIDIA Corporation",\n        "peers": ["AMD", "INTC"]\n    },\n    {\n        "ticker": "AMD",\n        "name": "Advanced Micro Devices, Inc.",\n        "peers": ["NVDA", "INTC"]\n    },\n    {\n        "ticker": "INTC",\n        "name": "Intel Corporation",\n        "peers": ["NVDA", "AMD"]\n    },\n    {\n        "ticker": "AAPL",\n        "name": "Apple Inc.",\n        "peers": ["MSFT", "GOOGL"]\n    },\n    {\n        "ticker": "GOOGL",\n        "name": "Alphabet Inc.",\n        "peers": ["META", "MSFT"]\n    }\n]\n'